# Session 18 course project: five builds from first principles

KLAi | Mathematics for AI and Machine Learning | **Your project. Automatic self-checks, no marks**

You build five things with numpy only: **(1) linear regression**, **(2) logistic regression**, **(3) PCA**, **(4) a small neural network with backpropagation** and **(5) scaled dot-product attention**. Each build is one or more **functions with fixed names**, which you write in the cells marked "your code here".

**How the self-checks work.** There is no exam and there are no marks. Type a student ID in the first code cell (any text you like, even your own name). It only creates a personal dataset, so your numbers are your own and nobody else has the same ones. It is not a test and nothing is sent anywhere. The checks call YOUR functions on YOUR data and on a second dataset, and compare the results with a reference. Each check shows OK or NOT YET, so you can see which build still needs work. No expected number is printed, so the way to see OK is to write a function that really computes the answer. Each check names its build. The last cell says "All N checks pass" when everything is right.

**Rules.** Use numpy only. Run the cells in order from the top. You may look at your notes, the slides and earlier notebooks, and you may talk with others and ask an assistant to explain an idea. The project is yours to build and keep. The spreadsheet version is a guided workbook with fixed toy numbers and its own checks.

In [ ]:
STUDENT_ID = "PUT-YOUR-STUDENT-ID-HERE"      # type your student ID between the quotes, then run this cell

In [ ]:
# Automatic check helper. Run this cell once. You do not need to read it.
_results = {}   # label -> True or False; running a check again replaces its old result
def check(label, value, expected, tol=0.01):
    try:
        import numpy as _np
        ok = bool(_np.all(_np.abs(_np.asarray(value, dtype=float) - _np.asarray(expected, dtype=float)) <= tol))
    except Exception:
        ok = False
    _results[label] = ok
    print(("OK       " if ok else "NOT YET  ") + label)
def show_checks():
    n = len(_results); k = sum(_results.values())
    print("All %d checks pass" % n if k == n and n else "%d of %d checks pass" % (k, n))

In [ ]:
# Self-check helper: data generators, reference implementations and checks. Run this cell once. You do not need to read it.
import hashlib
import math
import warnings

import numpy as np

NAMES = ['fit_linear', 'fit_linear_gd', 'sigmoid', 'logistic_loss', 'fit_logistic', 'pca_first_component', 'pca_variance_share',
         'mlp_forward', 'mlp_backward', 'mlp_train', 'softmax_rows', 'attention']
BUILD_NAMES = {1: 'Build 1 (linear regression)', 2: 'Build 2 (logistic regression)', 3: 'Build 3 (PCA)', 4: 'Build 4 (neural network)', 5: 'Build 5 (attention)'}


def seed_from_id(student_id):
    """A stable seed from the student ID text: capital letters, single spaces, then SHA-256 (the same on every computer)."""
    text = ' '.join(str(student_id).split()).upper()
    return int.from_bytes(hashlib.sha256(text.encode('utf-8')).digest()[:8], 'big')


def make_datasets(seed):
    """Every dataset of the course project for one seed, from numpy default_rng streams."""
    rngs = [np.random.default_rng(s) for s in np.random.SeedSequence(seed).spawn(5)]
    D = {'seed': seed}
    # 1. regression: 8 points on a noisy line
    r = rngs[0]
    n = 8
    x = np.sort(np.round(r.uniform(1, 10, n), 1))
    w0, w1 = round(float(r.uniform(0.5, 3)), 2), round(float(r.uniform(0.5, 2.5)), 2)
    y = np.round(w0 + w1 * x + r.normal(0, 0.8, n), 2)
    D['lin'] = dict(x=x, y=y, X=np.column_stack([np.ones(n), x]), X2=np.column_stack([np.ones(n), x, x ** 2]))
    # 2. logistic: 40 students, hours of study (0 to 10) and pass or fail
    r = rngs[1]
    n = 40
    x = np.round(r.uniform(0, 10, n), 1)
    c = r.uniform(4, 6)
    yb = (r.uniform(size=n) < 1 / (1 + np.exp(-0.9 * (x - c)))).astype(float)
    yb[np.argmin(x)], yb[np.argmax(x)] = 0.0, 1.0
    D['log'] = dict(x=x, y=yb, X=np.column_stack([np.ones(n), x]))
    # 3. PCA: 30 points in a tilted, off-centre cloud
    r = rngs[2]
    n = 30
    th = r.uniform(0.3, 1.2)
    R = np.array([[np.cos(th), -np.sin(th)], [np.sin(th), np.cos(th)]])
    P = (r.normal(size=(n, 2)) * np.array([2.0, 0.6])) @ R.T + r.uniform(5, 12, 2)
    D['pca'] = dict(P=np.round(P, 3))
    # 4. XOR-style network data: 10 jittered points around each corner of the unit square
    r = rngs[3]
    corners = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=float)
    Xn = np.repeat(corners, 10, axis=0) + r.normal(0, 0.1, (40, 2))
    yn = np.repeat(np.array([0.0, 1.0, 1.0, 0.0]), 10)
    D['xor'] = dict(X=np.round(Xn, 3), y=yn)
    # 5. attention: four tokens with 3-number embeddings, three weight matrices, and a wider 6-number example
    r = rngs[4]
    D['att'] = dict(X=np.round(r.normal(size=(4, 3)), 1), Wq=np.round(r.normal(size=(3, 3)), 1), Wk=np.round(r.normal(size=(3, 3)), 1), Wv=np.round(r.normal(size=(3, 3)), 1),
                    Q2=np.round(r.normal(0, 2, (3, 6)), 1), K2=np.round(r.normal(0, 2, (3, 6)), 1), V2=np.round(r.normal(0, 2, (3, 6)), 1))
    return D


# ---- trusted reference implementations 
def ref_fit_linear(X, y):
    return np.linalg.lstsq(X, y, rcond=None)[0]


def ref_fit_linear_gd(X, y, lr, steps):
    w = np.zeros(X.shape[1])
    for _ in range(int(steps)):
        w = w - lr * (2 / len(y)) * (X.T @ (X @ w - y))
    return w


def ref_sigmoid(z):
    return 1 / (1 + np.exp(-np.asarray(z, dtype=float)))


def ref_logistic_loss(w, X, y):
    p = ref_sigmoid(X @ w)
    return float(-np.mean(y * np.log(p) + (1 - y) * np.log(1 - p)))


def ref_fit_logistic(X, y, lr, steps):
    w = np.zeros(X.shape[1])
    for _ in range(int(steps)):
        w = w - lr * (X.T @ (ref_sigmoid(X @ w) - y)) / len(y)
    return w


def ref_pca(P):
    Pc = P - P.mean(axis=0)
    vals, vecs = np.linalg.eigh(Pc.T @ Pc / len(P))
    return vecs[:, -1], float(vals[-1] / vals.sum())


def ref_mlp_init(d, hidden, seed):
    rng = np.random.default_rng(seed)
    W1 = rng.normal(0, 0.5, (hidden, d))
    W2 = rng.normal(0, 0.5, (1, hidden))
    return W1, np.zeros(hidden), W2, np.zeros(1)


def ref_mlp_forward(W1, b1, W2, b2, X):
    h = np.tanh(X @ W1.T + b1)
    return h, ref_sigmoid(h @ W2.T + b2)[:, 0]


def ref_mlp_loss(W1, b1, W2, b2, X, y):
    p = ref_mlp_forward(W1, b1, W2, b2, X)[1]
    return float(-np.mean(y * np.log(p) + (1 - y) * np.log(1 - p)))


def ref_mlp_backward(W1, b1, W2, b2, X, y):
    h, p = ref_mlp_forward(W1, b1, W2, b2, X)
    dz = ((p - y) / len(y))[:, None]
    dh = (dz @ W2) * (1 - h ** 2)
    return dh.T @ X, dh.sum(axis=0), dz.T @ h, dz.sum(axis=0)


def ref_mlp_train(X, y, hidden, lr, steps, seed):
    W1, b1, W2, b2 = ref_mlp_init(X.shape[1], hidden, seed)
    for _ in range(int(steps)):
        g = ref_mlp_backward(W1, b1, W2, b2, X, y)
        W1, b1, W2, b2 = W1 - lr * g[0], b1 - lr * g[1], W2 - lr * g[2], b2 - lr * g[3]
    return W1, b1, W2, b2


def numeric_mlp_gradient(params, X, y, eps=1e-6):
    """Central finite differences of the loss for every parameter. Independent of any backpropagation code."""
    out = []
    for k, arr in enumerate(params):
        g = np.zeros_like(arr, dtype=float)
        for idx in np.ndindex(arr.shape):
            up = [a.copy() for a in params]
            dn = [a.copy() for a in params]
            up[k][idx] += eps
            dn[k][idx] -= eps
            g[idx] = (ref_mlp_loss(*up, X, y) - ref_mlp_loss(*dn, X, y)) / (2 * eps)
        out.append(g)
    return out


def ref_softmax_rows(S):
    S = np.asarray(S, dtype=float)
    e = np.exp(S - S.max(axis=1, keepdims=True))
    return e / e.sum(axis=1, keepdims=True)


def ref_attention(Q, K, V):
    w = ref_softmax_rows(Q @ K.T / math.sqrt(K.shape[1]))
    return w, w @ V


# ---- the checks. Each takes (D, L): D = datasets for one seed, L = dict of the learner's functions. It returns True or False (an exception counts as False).
def _same(a, b, tol=1e-6):
    a, b = np.asarray(a, dtype=float), np.asarray(b, dtype=float)
    return a.size == b.size and bool(np.all(np.isfinite(a))) and bool(np.allclose(a.ravel(), b.ravel(), atol=tol, rtol=1e-6))


def _same_direction(v, ref, tol=1e-6):
    v = np.asarray(v, dtype=float).ravel()
    return v.size == ref.size and bool(np.all(np.isfinite(v))) and abs(np.linalg.norm(v) - 1) < 1e-6 and abs(abs(float(v @ ref)) - 1) < tol


def c_lin_weights(D, L):
    return _same(L['fit_linear'](D['lin']['X'], D['lin']['y']), ref_fit_linear(D['lin']['X'], D['lin']['y']))


def c_lin_three_columns(D, L):
    return _same(L['fit_linear'](D['lin']['X2'], D['lin']['y']), ref_fit_linear(D['lin']['X2'], D['lin']['y']), 1e-5)


def c_lin_gd(D, L):
    return _same(L['fit_linear_gd'](D['lin']['X'], D['lin']['y'], 0.005, 400), ref_fit_linear_gd(D['lin']['X'], D['lin']['y'], 0.005, 400))


def c_lin_gd_other(D, L):
    return _same(L['fit_linear_gd'](D['lin']['X'], D['lin']['y'], 0.002, 60), ref_fit_linear_gd(D['lin']['X'], D['lin']['y'], 0.002, 60))


def c_sigmoid(D, L):
    z = np.concatenate([[-12.0, 15.0], D['log']['x'][:6] - 5.0])      # depends on the dataset, so a typed-in answer cannot pass
    return _same(L['sigmoid'](z), ref_sigmoid(z), 1e-9)


def c_log_loss(D, L):
    w = np.array([-1.3, 0.35])
    return _same(L['logistic_loss'](w, D['log']['X'], D['log']['y']), ref_logistic_loss(w, D['log']['X'], D['log']['y']), 1e-9)


def c_log_fit(D, L):
    X, y = D['log']['X'], D['log']['y']
    return _same(L['fit_logistic'](X, y, 0.1, 30), ref_fit_logistic(X, y, 0.1, 30), 1e-7)


def c_log_fit_other(D, L):
    X, y = D['log']['X'], D['log']['y']
    return _same(L['fit_logistic'](X, y, 0.03, 400), ref_fit_logistic(X, y, 0.03, 400), 1e-6)


def c_pca_direction(D, L):
    return _same_direction(L['pca_first_component'](D['pca']['P']), ref_pca(D['pca']['P'])[0])


def c_pca_share(D, L):
    return _same(L['pca_variance_share'](D['pca']['P']), ref_pca(D['pca']['P'])[1], 1e-6)


def c_pca_shift(D, L):
    return _same_direction(L['pca_first_component'](D['pca']['P'] + np.array([100.0, -50.0])), ref_pca(D['pca']['P'])[0])


def _params(D):
    return ref_mlp_init(2, 4, 1000 + D['seed'] % 1000)


def c_mlp_forward(D, L):
    prm = _params(D)
    X = D['xor']['X']
    h, p = L['mlp_forward'](*prm, X)
    rh, rp = ref_mlp_forward(*prm, X)
    return _same(h, rh, 1e-8) and _same(p, rp, 1e-8)


def c_mlp_grad_out(D, L):
    prm = _params(D)
    g = L['mlp_backward'](*prm, D['xor']['X'], D['xor']['y'])
    num = numeric_mlp_gradient(prm, D['xor']['X'], D['xor']['y'])
    return _same(g[2], num[2], 1e-6) and _same(g[3], num[3], 1e-6)


def c_mlp_grad_hidden(D, L):
    prm = _params(D)
    g = L['mlp_backward'](*prm, D['xor']['X'], D['xor']['y'])
    num = numeric_mlp_gradient(prm, D['xor']['X'], D['xor']['y'])
    return _same(g[0], num[0], 1e-6) and _same(g[1], num[1], 1e-6)


def c_mlp_train(D, L):
    X, y = D['xor']['X'], D['xor']['y']
    seed = D['seed'] % 10 ** 6
    got = L['mlp_train'](X, y, 4, 0.5, 300, seed)
    ref = ref_mlp_train(X, y, 4, 0.5, 300, seed)
    return _same(ref_mlp_forward(*got, X)[1], ref_mlp_forward(*ref, X)[1], 1e-5)


def c_mlp_train_other(D, L):
    X, y = D['xor']['X'], D['xor']['y']
    seed = (D['seed'] + 7) % 10 ** 6
    got = L['mlp_train'](X, y, 3, 0.3, 150, seed)
    ref = ref_mlp_train(X, y, 3, 0.3, 150, seed)
    return _same(ref_mlp_forward(*got, X)[1], ref_mlp_forward(*ref, X)[1], 1e-5)


def c_softmax(D, L):
    S = np.vstack([[1000.0, 1001.0, 1002.0], D['att']['X'][0] * 3, D['att']['X'][1]])      # one row of very large scores, two rows from the dataset
    out = np.asarray(L['softmax_rows'](S), dtype=float)
    return _same(out, ref_softmax_rows(S), 1e-9) and _same(out.sum(axis=1), np.ones(3), 1e-9)


def _qkv(D):
    a = D['att']
    return a['X'] @ a['Wq'], a['X'] @ a['Wk'], a['X'] @ a['Wv']


def c_att_weights(D, L):
    Q, K, V = _qkv(D)
    return _same(L['attention'](Q, K, V)[0], ref_attention(Q, K, V)[0], 1e-8)


def c_att_output(D, L):
    Q, K, V = _qkv(D)
    return _same(L['attention'](Q, K, V)[1], ref_attention(Q, K, V)[1], 1e-8)


def c_att_wide(D, L):
    a = D['att']
    got = L['attention'](a['Q2'], a['K2'], a['V2'])
    ref = ref_attention(a['Q2'], a['K2'], a['V2'])
    return _same(got[0], ref[0], 1e-8) and _same(got[1], ref[1], 1e-8)


# (build, id, label, function). There are no marks: each check shows OK or NOT YET so that you can see what still needs work.
CHECKS = [
    (1, 'lin_weights', 'fit_linear gives the least squares weights', c_lin_weights),
    (1, 'lin_three_columns', 'fit_linear works with three columns in X', c_lin_three_columns),
    (1, 'lin_gd', 'fit_linear_gd (learning rate 0.005, 400 steps) follows the same path', c_lin_gd),
    (1, 'lin_gd_other', 'fit_linear_gd uses the learning rate and steps it is given (0.002, 60)', c_lin_gd_other),
    (2, 'sigmoid', 'sigmoid', c_sigmoid),
    (2, 'log_loss', 'logistic_loss (mean cross-entropy)', c_log_loss),
    (2, 'log_fit', 'fit_logistic after 30 steps (learning rate 0.1)', c_log_fit),
    (2, 'log_fit_other', 'fit_logistic with other settings (0.03, 400 steps)', c_log_fit_other),
    (3, 'pca_direction', 'pca_first_component gives the first principal component', c_pca_direction),
    (3, 'pca_share', 'pca_variance_share gives the share of variance on it', c_pca_share),
    (3, 'pca_shift', 'pca_first_component is unchanged when all points are shifted (centring)', c_pca_shift),
    (4, 'mlp_forward', 'mlp_forward: hidden activations and outputs', c_mlp_forward),
    (4, 'mlp_grad_out', 'mlp_backward: gradients of W2 and b2 match a numeric gradient', c_mlp_grad_out),
    (4, 'mlp_grad_hidden', 'mlp_backward: gradients of W1 and b1 match a numeric gradient', c_mlp_grad_hidden),
    (4, 'mlp_train', 'mlp_train (4 hidden units, 300 steps) reaches the same network', c_mlp_train),
    (4, 'mlp_train_other', 'mlp_train with other settings (3 hidden units, 150 steps)', c_mlp_train_other),
    (5, 'softmax', 'softmax_rows, including very large scores', c_softmax),
    (5, 'att_weights', 'attention weights for your tokens', c_att_weights),
    (5, 'att_output', 'attention output for your tokens', c_att_output),
    (5, 'att_wide', 'attention on 6-number keys (the square root of d_k)', c_att_wide),
]


def run_check(fn, D, L):
    """True or False. Any error, NaN or missing function is False."""
    try:
        with warnings.catch_warnings(), np.errstate(all='ignore'):
            warnings.simplefilter('ignore')
            return bool(fn(D, L))
    except BaseException as e:  # noqa: BLE001  (a learner function may raise anything, including a time-out)
        if isinstance(e, KeyboardInterrupt):
            raise
        return False

PLACEHOLDER = "PUT-YOUR-STUDENT-ID-HERE"
HIDDEN = make_datasets(20260518)      # a second small dataset that only the checks use

def run_build(k):
    """Runs the checks of build k on YOUR data and on a second dataset, calling YOUR functions."""
    L = {n: globals().get(n) for n in NAMES}
    for b, cid, label, fn in CHECKS:
        if b == k:
            for tag, D in (("your data", DATA), ("second dataset", HIDDEN)):
                check(BUILD_NAMES[b] + ": " + label + ", " + tag, run_check(fn, D, L), True, 0)

def show(label, thunk):
    """Prints thunk() or says why it does not work yet (so an unfinished function does not stop the notebook)."""
    try:
        print(label + ":", thunk())
    except Exception as e:
        print(label + ": not working yet (" + type(e).__name__ + ")")

In [ ]:
SEED = seed_from_id(STUDENT_ID)
DATA = make_datasets(SEED)            # all your datasets, generated from your ID
lin, logd, pca_data, xor_data, att_data = DATA["lin"], DATA["log"], DATA["pca"], DATA["xor"], DATA["att"]
check("Setup: a student ID was entered", STUDENT_ID.strip().upper() != PLACEHOLDER and len(STUDENT_ID.strip()) > 2, True, 0)
print("student ID:", STUDENT_ID, "  your seed:", SEED)

---
## Step 1: Build 1: linear regression (normal equation and gradient descent)

Eight readings: hours of use x and energy used y in kWh. The model is y-hat = w0 + w1 x. The design matrix X has a column of ones first (for w0) and the hours second. Write two functions that return the weights w.

In [ ]:
X, y = lin["X"], lin["y"]                  # your design matrix (8 rows, 2 columns) and your energy readings
print("hours of use:", lin["x"])
print("energy used (kWh):", y)

**PREDICT first.** Look at your own numbers. Roughly what slope w1 do you expect? (How much does y rise per extra hour?)

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
# fit_linear(X, y): the weights that minimise the sum of squared errors, in one step (Build 1 slide). fit_linear_gd(X, y, lr, steps): the same weights by repeated small steps from w = 0
def fit_linear(X, y):
    return None  # your code here

def fit_linear_gd(X, y, lr, steps):
    return None  # your code here

In [ ]:
show("normal equation weights (w0, w1)", lambda: fit_linear(X, y))
show("gradient descent, lr 0.005, 400 steps", lambda: fit_linear_gd(X, y, 0.005, 400))
show("your mean squared error", lambda: np.mean((X @ fit_linear(X, y) - y) ** 2))
run_build(1)

**CHANGE one thing.** Call `fit_linear_gd(X, y, 0.05, 400)` in a new cell. What happens to the weights, and why?

> What I saw: 

---
## Step 2: Build 2: logistic regression (gradient descent)

Forty students: hours of study x and y = 1 if they passed. The model gives p = sigmoid(w0 + w1 x). The loss is the mean binary cross-entropy from Session 16. Write the sigmoid, the loss and the gradient descent function.

In [ ]:
Xl, yl = logd["X"], logd["y"]
print("hours of study (first 10):", logd["x"][:10])
print("passed (first 10):", yl[:10], "  share who passed:", yl.mean())

**PREDICT first.** The gradient starts from w = 0, where every predicted probability is 0.5. Will w1 move up or down in the first step?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
# sigmoid(z); logistic_loss(w, X, y) = mean binary cross-entropy; fit_logistic(X, y, lr, steps): gradient descent from w = 0 (Build 2 slide)
def sigmoid(z):
    return None  # your code here

def logistic_loss(w, X, y):
    return None  # your code here

def fit_logistic(X, y, lr, steps):
    return None  # your code here

In [ ]:
show("loss at w = 0 (should be ln 2 = 0.693)", lambda: logistic_loss(np.zeros(2), Xl, yl))
show("weights after 2000 steps (lr 0.1)", lambda: fit_logistic(Xl, yl, 0.1, 2000))
show("training accuracy", lambda: np.mean((sigmoid(Xl @ fit_logistic(Xl, yl, 0.1, 2000)) > 0.5) == (yl == 1)))
run_build(2)

**READ the output.** Is the model confident (probabilities near 0 or 1)? Why does the overlap between passes and fails stop it from being sure?

> My answer: 

---
## Step 3: Build 3: PCA via the covariance eigen-decomposition

Thirty points in two dimensions, in a tilted cloud that is not centred on zero. PCA centres the data, forms the covariance matrix (divide by n, the course convention) and finds its eigenvectors. The eigenvector with the largest eigenvalue is the first principal component. The sign of an eigenvector is arbitrary, so the checks accept either sign.

In [ ]:
P = pca_data["P"]                         # 30 rows, 2 columns
print("column means:", P.mean(axis=0))

**PREDICT first.** Look at the first few rows of P. Do the two columns rise together? Which way will the first principal component point: along (1, 1), along (1, -1), or along (1, 0)?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
# pca_first_component(P): the unit vector of largest variance. pca_variance_share(P): the first eigenvalue divided by the sum of both (Build 3 slides)
def pca_first_component(P):
    return None  # your code here

def pca_variance_share(P):
    return None  # your code here

In [ ]:
show("first principal component", lambda: pca_first_component(P))
show("share of the variance on it", lambda: pca_variance_share(P))
run_build(3)

**READ the output.** If you keep only the position of each point along the first component (one number per point), how much of the spread do you keep?

> My answer: 

---
## Step 4: Build 4: a small neural network with backpropagation

Forty points around the corners of the unit square. The label is XOR: 1 when exactly one of the two inputs is near 1. The network has two inputs, a hidden layer of tanh units and one sigmoid output; the loss is the mean binary cross-entropy. The starting weights are drawn by `np.random.default_rng(seed)` in this order: W1 = normal(0, 0.5, (hidden, 2)), then W2 = normal(0, 0.5, (1, hidden)); both biases start at 0. Shapes: W1 is (hidden, 2), b1 is (hidden,), W2 is (1, hidden), b2 is (1,). Write the forward pass, the backward pass and the training function.

In [ ]:
Xn, yn = xor_data["X"], xor_data["y"]
print("first rows of X:", Xn[:3].tolist(), "...  labels:", yn[[0, 10, 20, 30]])

**PREDICT first.** Before any training, will the network give answers close to 0 and 1, or all near 0.5?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
# mlp_forward returns (h, p); mlp_backward returns (gW1, gb1, gW2, gb2) for the mean loss; mlp_train(X, y, hidden, lr, steps, seed) returns (W1, b1, W2, b2) (Build 4 slides)
def mlp_forward(W1, b1, W2, b2, X):
    return None, None  # your code here: return h, p

def mlp_backward(W1, b1, W2, b2, X, y):
    return None  # your code here: return gW1, gb1, gW2, gb2

def mlp_train(X, y, hidden, lr, steps, seed):
    return None  # your code here: return W1, b1, W2, b2

In [ ]:
def mean_loss(p, y):
    return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))

def show_training():
    start = mlp_train(Xn, yn, 4, 0.5, 0, SEED % 1000000)
    end = mlp_train(Xn, yn, 4, 0.5, 3000, SEED % 1000000)
    print("loss before training:", mean_loss(mlp_forward(*start, Xn)[1], yn))
    print("loss after 3000 steps:", mean_loss(mlp_forward(*end, Xn)[1], yn))
    print("rounded outputs for the four corners:", np.round(mlp_forward(*end, Xn)[1][[0, 10, 20, 30]]))
show("training", show_training)
run_build(4)

**READ the output.** XOR cannot be solved by logistic regression on the two inputs. What did the hidden layer add?

> My answer: 

---
## Step 5: Build 5: scaled dot-product attention

Four tokens with 3-number embeddings (your data). Queries, keys and values come from three weight matrices. The attention weights are softmax(Q K^T / sqrt(d_k)) taken along each row, and the output is the weights times V. Write the row-wise softmax and the attention function. attention returns (weights, output).

In [ ]:
Xt, Wq, Wk, Wv = att_data["X"], att_data["Wq"], att_data["Wk"], att_data["Wv"]
Q, K, V = Xt @ Wq, Xt @ Wk, Xt @ Wv
print("token embeddings:", Xt.tolist())
print("shapes of Q, K, V:", Q.shape, K.shape, V.shape)

**PREDICT first.** Every row of the attention weights is a probability distribution over the four tokens. What must each row add up to?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
# softmax_rows(S): a softmax along each row that survives scores like 1000. attention(Q, K, V): returns (weights, output) (Build 5 slides)
def softmax_rows(S):
    return None  # your code here

def attention(Q, K, V):
    return None, None  # your code here: return weights, output

In [ ]:
show("attention weights", lambda: np.round(attention(Q, K, V)[0], 3).tolist())
show("row sums", lambda: attention(Q, K, V)[0].sum(axis=1))
# why divide by sqrt(d_k)? dot products of long random vectors grow with the length (given code, no check)
rng_demo = np.random.default_rng(SEED)
qs, ks = rng_demo.normal(size=(2000, 64)), rng_demo.normal(size=(2000, 64))
print("standard deviation of q . k for length 64:", np.std(np.sum(qs * ks, axis=1)), " after dividing by sqrt(64):", np.std(np.sum(qs * ks, axis=1)) / 8)
run_build(5)

**CHANGE one thing.** Change `attention` so it does not divide by the square root of d_k and compare the weights on the 6-number example `att_data["Q2"]`, `att_data["K2"]`. Which rows get sharper (closer to one-hot)?

> What I saw: 

---
## Save your work

Run every cell again from the top (Runtime, Restart and run all). The last line must say all checks pass. Save the notebook and keep it: it is your course project. There is nothing to hand in for marks. If you would like feedback, show the notebook to a classmate or to your facilitator and use the project checklist below.

## Make it yours (optional)

Take the same functions to data you care about. Two examples: put two columns of your own numbers in a list and call `fit_linear` to fit a line, or call `pca_first_component` on two columns of a small table from your work. Never use private or confidential data in a shared notebook. Then go through the checklist, alone or with a partner.

## Project checklist (for you and a peer)

- [ ] Every self-check says OK, or I can explain what NOT YET still means.
- [ ] I can explain Build 1 to someone in two sentences, with no formulas.
- [ ] I can say why the logistic loss is cross-entropy (Session 16).
- [ ] I can say what the first principal component is, in plain words.
- [ ] I checked my backpropagation gradient against a numeric gradient.
- [ ] I can say why attention divides by the square root of d_k.
- [ ] My notebook runs from the top with Restart and run all.
- [ ] I tried at least one change (a new learning rate, my own data or no scaling) and wrote down what happened.

In [ ]:
show_checks()